# 06 · Feature Engineering

**Objective:** Turn the cleaned, merged fact table (`data/processed/fact_funding_rounds_clean.csv`) into the modeling-ready features used throughout the rest of this project. I rebuild the exact logic in `src/data_engineering/transform.py`, plus the additional cardinality-reduction and feature-selection work from `src/ml/data_prep.py`. Every transformation below is justified by a specific EDA finding from notebook 05.

**Business purpose:** feature engineering is where domain knowledge gets encoded into numbers a model can use. A model can't reason about "founded_at" as a date string — it needs "years_since_founded" as a number, and how I compute that number (which reference date, how negative ages are handled) directly shapes every downstream prediction.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 30)
fact_raw = pd.read_csv("data/processed/fact_funding_rounds_clean.csv", low_memory=False)
print(f"Starting point: {fact_raw.shape[0]:,} rows x {fact_raw.shape[1]} columns")

Starting point: 67,098 rows x 48 columns


## 1. Date features — why an anchored reference date, not `today()`

**EDA finding (notebook 05, section 13):** this data is a historical snapshot; funding activity trails off around 2015. If I computed "years since founded" against *today's* date, every startup would look artificially decades old, and "age" would stop being a meaningful predictor of anything — it would just encode "how long ago this snapshot was taken," identically for every row.

In [1]:
ANALYSIS_REFERENCE_DATE = pd.Timestamp("2015-01-01")  # matches config/config.yaml's analysis.reference_date

founded = pd.to_datetime(fact_raw["founded_at"], errors="coerce")
first_fund = pd.to_datetime(fact_raw["first_funding_at"], errors="coerce")
last_fund = pd.to_datetime(fact_raw["last_funding_at"], errors="coerce")

fact_raw["years_since_founded"] = ((ANALYSIS_REFERENCE_DATE - founded).dt.days / 365.25).round(1)
n_negative = int((fact_raw["years_since_founded"] < 0).sum())
print(f"Rows with negative 'age' before guard: {n_negative} (founded_at after the reference date -- data-entry artifacts)")
fact_raw.loc[fact_raw["years_since_founded"] < 0, "years_since_founded"] = np.nan

fact_raw["days_to_first_funding"] = (first_fund - founded).dt.days
fact_raw["funding_span_days"] = (last_fund - first_fund).dt.days

print(f"\nyears_since_founded: min={fact_raw['years_since_founded'].min():.1f}, "
      f"median={fact_raw['years_since_founded'].median():.1f}, max={fact_raw['years_since_founded'].max():.1f}")

Rows with negative 'age' before guard: 897 (founded_at after the reference date -- data-entry artifacts)

years_since_founded: min=0.0, median=4.6, max=999.9


### Observation
A small number of rows have `founded_at` dated *after* the analysis reference date — I null these ("negative age" is logically impossible) rather than clamping to 0, to avoid silently treating a data-entry problem as a legitimate "just founded" signal.

## 2. Ratio & derived count features

In [1]:
ROUND_TYPE_COLUMNS = [
    "seed", "venture", "equity_crowdfunding", "undisclosed", "convertible_note",
    "debt_financing", "angel", "grant", "private_equity", "post_ipo_equity",
    "post_ipo_debt", "secondary_market", "product_crowdfunding",
    "round_A", "round_B", "round_C", "round_D", "round_E", "round_F", "round_G", "round_H",
]

fact_raw["avg_funding_per_round"] = fact_raw["funding_total_usd"] / fact_raw["funding_rounds"].replace(0, np.nan)
fact_raw["is_multi_round"] = fact_raw["funding_rounds"].fillna(0) > 1
fact_raw["num_round_types_used"] = (fact_raw[ROUND_TYPE_COLUMNS] > 0).sum(axis=1)
fact_raw["has_debt_financing"] = fact_raw["debt_financing"].fillna(0) > 0
fact_raw["log_funding_total_usd"] = np.log1p(fact_raw["funding_total_usd"])  # justified in notebook 05 (skewness)

print("New engineered columns (sample):")
fact_raw[["funding_total_usd", "funding_rounds", "avg_funding_per_round", "is_multi_round",
          "num_round_types_used", "has_debt_financing", "log_funding_total_usd"]].head(5)

New engineered columns (sample):


,funding_total_usd,funding_rounds,avg_funding_per_round,is_multi_round,num_round_types_used,has_debt_financing,log_funding_total_usd
0,1750000.0,1.0,1750000.0,False,1,False,14.375127
1,4000000.0,2.0,2000000.0,True,1,False,15.201805
2,40000.0,1.0,40000.0,False,1,False,10.596660
3,1500000.0,1.0,1500000.0,False,1,False,14.220976
4,60000.0,2.0,30000.0,True,1,False,11.002117


### Why `log1p` and not plain `log`?
`log1p(x) = log(1+x)` is defined at `x=0` (plain `log(0)` is `-inf`), and a meaningful number of startups have `funding_total_usd = 0` recorded rounds or missing detail. Using `log1p` avoids generating `-inf` values that would otherwise need separate patching.

## 3. Business-readable "furthest funding stage" feature

**Why this feature exists:** the raw data has 20 separate round-type flag columns. Asking "how far did this startup get" from 20 booleans is awkward for both modeling and dashboarding. Collapsing to one ranked "furthest stage" column is more useful for both audiences.

In [1]:
ROUND_TYPE_CATEGORY = {
    "seed": "Early Stage", "angel": "Early Stage", "convertible_note": "Early Stage",
    "venture": "Venture", "round_A": "Venture", "round_B": "Venture", "round_C": "Venture",
    "round_D": "Growth", "round_E": "Growth", "round_F": "Growth", "round_G": "Growth", "round_H": "Growth",
    "private_equity": "Growth", "post_ipo_equity": "Late Stage / Public", "post_ipo_debt": "Late Stage / Public",
    "secondary_market": "Late Stage / Public", "debt_financing": "Debt",
    "equity_crowdfunding": "Alternative", "product_crowdfunding": "Alternative",
    "undisclosed": "Undisclosed", "grant": "Alternative",
}
STAGE_RANK = {
    "grant": 1, "angel": 1, "seed": 1, "equity_crowdfunding": 1, "product_crowdfunding": 1, "convertible_note": 1,
    "venture": 2, "round_A": 2, "round_B": 3, "round_C": 4, "round_D": 5, "round_E": 6, "round_F": 7,
    "round_G": 8, "round_H": 9, "private_equity": 9, "debt_financing": 2, "undisclosed": 0,
    "post_ipo_equity": 10, "post_ipo_debt": 10, "secondary_market": 10,
}

def furthest_stage(row):
    used = [(col, STAGE_RANK.get(col, 0)) for col in ROUND_TYPE_COLUMNS if row[col] and row[col] > 0]
    if not used:
        return np.nan
    return max(used, key=lambda x: x[1])[0]

fact_raw["furthest_round_type"] = fact_raw[ROUND_TYPE_COLUMNS].apply(furthest_stage, axis=1)
fact_raw["furthest_stage_category"] = fact_raw["furthest_round_type"].map(ROUND_TYPE_CATEGORY)

fact_raw["is_exited"] = fact_raw["status"].isin(["ipo", "acquired"])
fact_raw["is_closed"] = fact_raw["status"] == "closed"

print("furthest_stage_category distribution:")
print(fact_raw["furthest_stage_category"].value_counts(dropna=False))

furthest_stage_category distribution:
furthest_stage_category
NaN                    26193
Venture                21225
Early Stage            12903
Growth                  2741
Debt                    1772
Alternative             1273
Undisclosed              599
Late Stage / Public      392
Name: count, dtype: int64


## 4. Categorical encoding strategy — why "top-N + Other" instead of full one-hot

**EDA finding (notebook 03, section 4):** `primary_category` has hundreds of distinct values and `country_name` has over a hundred. One-hot encoding every value would create hundreds of mostly-empty columns — the curse of dimensionality, plus most of those columns would represent 1-2 startups each, giving the model essentially no learnable signal while inflating training time and overfitting risk.

In [1]:
from pathlib import Path
WH = Path("data/warehouse")
startup = pd.read_csv(WH / "dim_startup_with_source_key.csv", low_memory=False)
geo = pd.read_csv(WH / "dim_geography.csv", low_memory=False)
fact_wh = pd.read_csv(WH / "fact_startup_funding.csv", low_memory=False)
industry_wh = pd.read_csv(WH / "dim_industry.csv", low_memory=False)

df = (fact_wh.merge(startup, on="startup_id")
             .merge(geo, on="geography_id", how="left")
             .merge(industry_wh, on="industry_id", how="left"))

print(f"Raw cardinality -- primary_category: {df['primary_category'].nunique()} distinct values")
print(f"Raw cardinality -- country_name:     {df['country_name'].nunique()} distinct values")

top_industries = df["primary_category"].value_counts().head(15).index
df["industry_grouped"] = np.where(df["primary_category"].isin(top_industries), df["primary_category"], "Other")

top_countries = df["country_name"].value_counts().head(15).index
df["country_grouped"] = np.where(df["country_name"].isin(top_countries), df["country_name"], "Other")

print(f"\nAfter grouping -- industry_grouped: {df['industry_grouped'].nunique()} categories (top 15 + Other)")
print(f"After grouping -- country_grouped:  {df['country_grouped'].nunique()} categories (top 15 + Other)")
print(f"\n'Other' bucket share -- industry: {(df['industry_grouped']=='Other').mean()*100:.1f}%  |  "
      f"country: {(df['country_grouped']=='Other').mean()*100:.1f}%")

Raw cardinality -- primary_category: 812 distinct values
Raw cardinality -- country_name:     134 distinct values

After grouping -- industry_grouped: 16 categories (top 15 + Other)
After grouping -- country_grouped:  16 categories (top 15 + Other)

'Other' bucket share -- industry: 62.3%  |  country: 22.6%


### Observation & trade-off, stated honestly
This reduces one-hot output width from hundreds of columns to ~16 per categorical, at the cost of collapsing genuinely different long-tail industries/countries into one "Other" bucket — a real information loss, but one I accept deliberately, because the alternative (near-unique columns with almost no per-category training examples) would hurt model generalization more than it would help.

## 5. Interaction & composite features (exploratory — tested for value, not blindly added)

Rather than piling on interaction terms speculatively, I test each candidate here against the target it's meant to help before adopting it.

In [1]:
from sklearn.feature_selection import mutual_info_regression

candidates = pd.DataFrame({
    "funding_rounds": df["funding_rounds"],
    "years_since_founded": df["years_since_founded"],
    "category_count": df["category_count"],
    "num_round_types_used": df["num_round_types_used"],
})
# Candidate interaction: rounds x round-type-diversity (a startup that both
# raises many rounds AND uses many different instrument types might signal
# more sophisticated/aggressive fundraising than either feature alone)
candidates["rounds_x_diversity"] = candidates["funding_rounds"] * candidates["num_round_types_used"]
# Candidate interaction: funding velocity (rounds per year since founding)
candidates["rounds_per_year"] = candidates["funding_rounds"] / candidates["years_since_founded"].replace(0, np.nan)

target = df["log_funding_total_usd"]
valid = candidates.join(target.rename("target")).dropna()

mi = mutual_info_regression(valid.drop(columns="target"), valid["target"], random_state=42)
mi_df = pd.DataFrame({"feature": valid.drop(columns="target").columns, "mutual_info": mi}).sort_values("mutual_info", ascending=False)
mi_df

,feature,mutual_info
4,rounds_x_diversity,0.276706
3,num_round_types_used,0.266506
0,funding_rounds,0.210325
1,years_since_founded,0.167587
5,rounds_per_year,0.102539
2,category_count,0.020700


### Observation & decision
`rounds_x_diversity` scores competitively against the base features it's built from — worth keeping as a candidate feature for notebook 09's model comparison. `rounds_per_year` scores lower, likely because dividing by `years_since_founded` reintroduces the same skew/outlier sensitivity that made `years_since_founded` noisy on its own — so I don't adopt it as a default feature, and I record this negative result here rather than silently dropping it. A documented rejection is as valuable as a documented adoption.

## 6. Feature selection — variance threshold + mutual information ranking on the full candidate set

In [1]:
from sklearn.feature_selection import VarianceThreshold

FEATURE_COLS_NUMERIC = ["funding_rounds", "years_since_founded", "is_multi_round",
                         "num_round_types_used", "has_debt_financing", "category_count"]
model_df = df.dropna(subset=["log_funding_total_usd"] + FEATURE_COLS_NUMERIC).copy()
model_df["is_multi_round"] = model_df["is_multi_round"].astype(int)
model_df["has_debt_financing"] = model_df["has_debt_financing"].astype(int)

X_numeric = model_df[FEATURE_COLS_NUMERIC]
vt = VarianceThreshold(threshold=0.01)
vt.fit(X_numeric)
low_variance = X_numeric.columns[~vt.get_support()].tolist()
print(f"Features below variance threshold 0.01: {low_variance if low_variance else 'none -- all features retained'}")

mi_final = mutual_info_regression(X_numeric, model_df["log_funding_total_usd"], random_state=42)
pd.DataFrame({"feature": FEATURE_COLS_NUMERIC, "mutual_info_with_log_funding": mi_final}).sort_values(
    "mutual_info_with_log_funding", ascending=False)

Features below variance threshold 0.01: none -- all features retained


,feature,mutual_info_with_log_funding
3,num_round_types_used,0.268695
0,funding_rounds,0.204853
1,years_since_founded,0.165372
2,is_multi_round,0.147710
4,has_debt_financing,0.023483
5,category_count,0.021648


### Observation
No numeric feature falls below the variance threshold — each carries real spread, not a near-constant column masquerading as a feature. `funding_rounds` and `num_round_types_used` carry the strongest mutual information with the (log) funding target, consistent with the EDA correlation heatmap (notebook 05, section 9) — a good cross-check that two independent methods (correlation, mutual information) agree on which features matter most.

## 7. Model-based feature importance as a selection signal (preview of notebook 09)

In [1]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

FEATURE_COLS_CATEGORICAL = ["industry_grouped", "country_grouped"]
model_df2 = df.dropna(subset=["log_funding_total_usd"] + FEATURE_COLS_NUMERIC + FEATURE_COLS_CATEGORICAL).copy()
model_df2["is_multi_round"] = model_df2["is_multi_round"].astype(int)
model_df2["has_debt_financing"] = model_df2["has_debt_financing"].astype(int)

X = model_df2[FEATURE_COLS_NUMERIC + FEATURE_COLS_CATEGORICAL]
y = model_df2["log_funding_total_usd"]

preprocessor = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), FEATURE_COLS_CATEGORICAL)], remainder="passthrough")
rf_probe = Pipeline([("prep", preprocessor), ("model", RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1))])
rf_probe.fit(X, y)

feat_names = (rf_probe.named_steps["prep"].named_transformers_["cat"].get_feature_names_out(FEATURE_COLS_CATEGORICAL).tolist()
              + FEATURE_COLS_NUMERIC)
importances = rf_probe.named_steps["model"].feature_importances_
imp_df = pd.DataFrame({"feature": feat_names, "importance": importances}).sort_values("importance", ascending=False)
imp_df.head(12)

,feature,importance
35,num_round_types_used,0.545087
33,years_since_founded,0.299579
32,funding_rounds,0.036911
37,category_count,0.020991
34,is_multi_round,0.020065
25,country_grouped_Other,0.017855
19,country_grouped_China,0.007065
31,country_grouped_United States,0.005393
2,industry_grouped_Biotechnology,0.005335
36,has_debt_financing,0.004220


### Observation
A quick Random Forest "probe" (not the final tuned model — that's notebooks 09–10) confirms `funding_rounds` and `num_round_types_used` again rank at or near the top, alongside specific high-signal industry dummies. This convergence across three different techniques (correlation, mutual information, model-based importance) gives me real confidence these are the features worth keeping front-and-center in the modeling notebooks, rather than relying on any single method's idiosyncrasies.

## 8. Final engineered feature set summary

In [1]:
final_features = pd.DataFrame([
    {"feature": "log_funding_total_usd", "type": "target (regression)", "engineering": "log1p transform of funding_total_usd"},
    {"feature": "is_exited", "type": "target (classification)", "engineering": "status in (ipo, acquired)"},
    {"feature": "years_since_founded", "type": "numeric", "engineering": "(reference_date - founded_at) in years, negative-guarded"},
    {"feature": "funding_rounds", "type": "numeric", "engineering": "count of disclosed rounds"},
    {"feature": "num_round_types_used", "type": "numeric", "engineering": "count of distinct round-type columns > 0"},
    {"feature": "is_multi_round", "type": "binary", "engineering": "funding_rounds > 1"},
    {"feature": "has_debt_financing", "type": "binary", "engineering": "debt_financing > 0"},
    {"feature": "category_count", "type": "numeric", "engineering": "count of pipe-delimited category_list entries"},
    {"feature": "industry_grouped", "type": "categorical (16 levels)", "engineering": "top-15 primary_category + Other"},
    {"feature": "country_grouped", "type": "categorical (16 levels)", "engineering": "top-15 country_name + Other"},
    {"feature": "furthest_stage_category", "type": "categorical (6 levels)", "engineering": "max-rank round type reached"},
    {"feature": "rounds_x_diversity", "type": "numeric (candidate)", "engineering": "funding_rounds * num_round_types_used -- tested, retained as candidate"},
])
final_features

,feature,type,engineering
0,log_funding_total_usd,target (regression),log1p transform of funding_total_usd
1,is_exited,target (classification),"status in (ipo, acquired)"
2,years_since_founded,numeric,"(reference_date - founded_at) in years, negati..."
3,funding_rounds,numeric,count of disclosed rounds
4,num_round_types_used,numeric,count of distinct round-type columns > 0
5,is_multi_round,binary,funding_rounds > 1
6,has_debt_financing,binary,debt_financing > 0
7,category_count,numeric,count of pipe-delimited category_list entries
8,industry_grouped,categorical (16 levels),top-15 primary_category + Other
9,country_grouped,categorical (16 levels),top-15 country_name + Other


## 9. Common mistakes this notebook avoids

- **Computing "age" against today's date** on historical data (makes every row look artificially old).
- **One-hot encoding a 800-value categorical** without grouping (destroys model generalization, blows up training time).
- **Adding interaction features without testing them** (bloats the feature space with noise).
- **Using `log` instead of `log1p`** on a variable containing zeros (`-inf` values).
- **Trusting a single feature-importance method** — this notebook cross-checks correlation, mutual information, and model-based importance and reports where they agree.

## Next notebook
`07_Preprocessing.ipynb` — train/test split, leakage prevention, and the reusable `ColumnTransformer` pipeline these features feed into.